In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit,StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from utils import *
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import wandb

### Read Data

In [3]:
df = pd.read_csv('../../../data/creditcard.csv')

df = create_features(df)

scaler = StandardScaler()
df[['_log_amount']] = scaler.fit_transform(df[['_log_amount']])
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,1.123062,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,-1.115298,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,1.680981,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,1.008128,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,0.669117,0,1,0,0.0,1.0,1.0,0,1


In [4]:
features = df.drop(['Time','Amount','Class','Hour_from_start_mod24'], axis=1).columns.tolist()
print(features)

['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', '_log_amount', 'is_night_proxy', 'is_business_hours_proxy', 'hour_sin', 'hour_cos', 'time_diff', 'is_high_amount', 'is_rapid_transaction']


In [5]:
X = df[features]
y = df['Class']

print(X.shape, y.shape)

(283726, 36) (283726,)


In [6]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, 'Class')

X_train: (181584, 36) y_train: (181584,)
X_val: (45396, 36) y_val: (45396,)
X_test: (56746, 36) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


### Train Model 

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=7,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
xg_model = XGBClassifier(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda = 1.0,
    tree_method = "hist",
    scale_pos_weight=neg/max(pos, 1),
    eval_metric='aucpr',
    random_state=42
)

xg_model.fit(X_train,y_train)
rf_model.fit(X_train,y_train)

p_test_rf = rf_model.predict_proba(X_test)[:, 1]
p_test_xg = xg_model.predict_proba(X_test)[:, 1]
final_test = 0.1 * p_test_rf + 0.9 * p_test_xg

result_rf = log_eval(y_test, p_test_rf)
result_xg = log_eval(y_test, p_test_xg)
result_final = log_eval(y_test,final_test)

p_val_rf = rf_model.predict_proba(X_val)[:, 1]
p_val_xg = xg_model.predict_proba(X_val)[:, 1]
final_val = 0.1 * p_val_rf + 0.9 * p_val_xg

lr_model = LogisticRegression()

lr_model.fit(final_val.reshape(-1,1),y_val)

p_cal = lr_model.predict_proba(final_test.reshape(-1,1))[:,1]

result_cal = log_eval(y_test,p_cal)
rs_final_df = pd.DataFrame([
    {"model": "RF", **result_rf},
    {"model": "XGB", **result_xg},
    {"model": "RF + XGB", **result_final},
    {"model": "RF + XGB + LR", **result_cal}
])
rs_final_df



,model,threshold,Cost,ROC_AUC,PR_AUC,debiased_ece,adaptive_ece,Brier
0,RF,0.048,2540.0,0.980542,0.816606,0.001058,0.000902,0.000405
1,XGB,0.023,3170.0,0.975748,0.790634,0.000315,0.000401,0.000498
2,RF + XGB,0.029,3130.0,0.986618,0.798936,0.000256,0.000253,0.000478
3,RF + XGB + LR,0.001,3435.0,0.986618,0.798936,0.000449,0.000792,0.000449


In [9]:
a = thr_for_precision(y_test,p_test_rf)
print(evaluate(y_test,p_test_rf,a['threshold']))

{'threshold': 0.46294636198721156, 'precision': 0.9032258064516129, 'recall': 0.7567567567567568, 'f1': 0.8235294117647058, 'roc_auc': 0.9805416564927434, 'auprc': 0.8166061981780931, 'brier': 0.0004048957261082295, 'tp': 56, 'fp': 6, 'fn': 18, 'tn': 56666}


In [10]:
a = thr_for_precision(y_test,p_test_xg)
print(evaluate(y_test,p_test_xg,a['threshold']))

{'threshold': 0.9691570997238159, 'precision': 0.9, 'recall': 0.7297297297297297, 'f1': 0.8059701492537313, 'roc_auc': 0.9757478310467441, 'auprc': 0.7906337623801777, 'brier': 0.0004982924438081682, 'tp': 54, 'fp': 6, 'fn': 20, 'tn': 56666}


In [11]:
a = thr_for_precision(y_test,final_test)
print(evaluate(y_test,final_test,a['threshold']))

{'threshold': 0.9105790981643769, 'precision': 0.9016393442622951, 'recall': 0.7432432432432432, 'f1': 0.8148148148148148, 'roc_auc': 0.9866178731667862, 'auprc': 0.7989356528696205, 'brier': 0.0004783054624395905, 'tp': 55, 'fp': 6, 'fn': 19, 'tn': 56666}


In [12]:
a = thr_for_precision(y_test,p_cal)
print(evaluate(y_test,p_cal,a['threshold']))

{'threshold': 0.5803178902206321, 'precision': 0.9016393442622951, 'recall': 0.7432432432432432, 'f1': 0.8148148148148148, 'roc_auc': 0.9866178731667862, 'auprc': 0.7989356528696205, 'brier': 0.0004493022777285541, 'tp': 55, 'fp': 6, 'fn': 19, 'tn': 56666}


In [13]:
# final_val_proba = 0.6 * val_xgb_proba + 0.4 * p_val

# best_th, _ = thr_min_cost(y_val, final_val_proba)

# final_test_proba = 0.6 * test_xgb_proba + 0.4 * p_test
# y_test_pred = (final_test_proba >= best_th).astype(int)

# rs_rf = log_eval(y_test, final_test_proba)
# rs = log_eval(y_val, final_val_proba)

# val_df = pd.DataFrame([
#     {"model": "RF + XBG test", **rs_rf},
#     {"model": "RF + XGB val", **rs}
# ])


# val_df

In [14]:
# start_run('Random forest và Calibrated')

In [15]:
# log_metrics(result_cal)